<a href="https://colab.research.google.com/github/anishjagota/undergrad_ml_assignments/blob/main/02_data_scientist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demand Estimation and Market Analysis: Air Fryers
## Part 2 — Data Scientist (Demand Estimation)

We estimate a logit-style demand model for the cleaned air-fryer market using linear regression on log market shares. The model is

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}\,p_{bt} + \beta_{rating}\,r_{bt} + \sum_{\ell=1}^L \beta_\ell\,x_{bt\ell} + \epsilon_{bt}.
$$

We use one constant price coefficient shared across all brands and years (so we can later use the first-order pricing condition to back out unit costs). Year and brand fixed effects enter through `pd.get_dummies(..., drop_first=True)`. The dropped brand and year are the reference categories, so the dummy coefficients are read relative to them.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


## Load data and define feature columns

In [3]:
df = pd.read_csv('air_fryers_clean_brand_year.csv')

feature_cols = [
    'compact_share',
    'dual_basket_share',
    'oven_style_share',
    'rotisserie_share',
    'window_share',
]

df.head()


,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364


## Build the design matrix and fit the regression

Reference categories (dropped by `drop_first=True`): brand `chefman` and year `2019`.

In [4]:
y = df['log_brand_share']

brand_dummies = pd.get_dummies(df['brand'],            prefix='brand', drop_first=True, dtype=int)
year_dummies  = pd.get_dummies(df['year'].astype(str), prefix='year',  drop_first=True, dtype=int)

X = pd.concat(
    [df.loc[:, ['avg_price', 'avg_rating'] + feature_cols],
     brand_dummies,
     year_dummies],
    axis=1,
)

model = LinearRegression()
model = model.fit(X, y)

y_hat = model.predict(X)
r2 = r2_score(y, y_hat)

print(f'Model intercept: {model.intercept_}')
print(f'R-squared: {r2}')

coef_table = pd.DataFrame({
    'variable':    model.feature_names_in_,
    'coefficient': model.coef_,
})
coef_table


Model intercept: -13.30489160121093
R-squared: 0.763453950091436


,variable,coefficient
0,avg_price,-0.037668
1,avg_rating,0.287517
2,compact_share,9.815304
3,dual_basket_share,-9.509686
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
6,window_share,12.880298
7,brand_cosori,2.551946
8,brand_cuisinart,6.422436
9,brand_dash,0.176655


## Answers

### 1. Estimated price coefficient $\hat{\beta}_{price}$

$\hat{\beta}_{price} \approx -0.0377$.

### 2. Is it negative? Why is that important?

Yes. A negative price coefficient is exactly what demand theory predicts: holding rating, product features, brand identity, and year fixed, a brand with a higher price has lower demand and thus a smaller market share.

 It is also a precondition for the next stage of the analysis. The first-order pricing condition we use to back out unit costs requires a downward-sloping demand curve — if the estimated price coefficient were positive, $\hat{s}'_{bt}(p_{bt}) = \hat{\beta}_{price}\,s_{bt}(1-s_{bt})$ would be positive and the implied marginal-cost formula would produce nonsense (unit costs above price). So a negative price coefficient is both the economically expected sign and a sanity check that the model is usable for cost inference.

### 3. Which product features are associated with higher demand?

From the coefficient table, the features with positive associations are:

- `window_share` ($\approx +12.9$): largest positive feature effect
- `compact_share` ($\approx +9.8$)
- `oven_style_share` ($\approx +1.9$)
- `avg_rating` ($\approx +0.29$): a one-point higher average rating is associated with about a 29% higher share, but ratings move so little across this dataset that this is a small force in practice.

However, many of these features should be read as correlational artifacts of the small sample rather than as causal feature effects.

### 4. Largest brand dummies (relative to dropped brand `chefman`)

Ranked from largest to smallest:

1. `cuisinart`   $\approx +6.42$
2. `ninja`       $\approx +5.84$
3. `instant_pot` $\approx +4.63$
4. `gowise usa`  $\approx +3.94$
5. `oster`       $\approx +3.93$
6. `nuwave`      $\approx +3.54$
7. `cosori`      $\approx +2.55$
8. `ultrean`     $\approx +0.94$
9. `dash`        $\approx +0.18$

Each coefficient is the brand's residual log-share advantage over Chefman after controlling for price, rating, and product features. The fact that Cuisinart's dummy is the largest is striking: even though Cuisinart has a small share, it has a very high price, and the model is essentially saying that if Cuisinart priced where Chefman prices, it would have a much larger share. So brand dummies should be interpreted as brand strength net of price.

### 5. Largest year dummies (relative to dropped year 2019)

All year dummies are small in absolute value:

- 2020: $\approx +0.12$
- 2021: $\approx +0.04$
- 2022: $\approx -0.10$
- 2023: $\approx -0.003$

There is no strong year-over-year trend in the average residual log share. This is consistent with the EDA finding that aggregate price and rating levels were stable over the 5 years; what changed was the redistribution of share among brands, not the overall market structure. The year fixed effects mostly absorb the multinomial-logit denominator and are not the main story.

### 6. Model $R^2$

$R^2 \approx 0.76$. The model explains about 76% of the variation in log brand share across the 50 brand-year cells. That is somewhat a reasonable fit for a parsimonious demand specification with a single price slope, two-way fixed effects, and five product-characteristic shares.